# Görev E — Taşıma müfredatı: "öğrendiği mesafede hatırlar" testi

**Zincir:** §17 bellek >1024'te çöktü → §15h sebep retention yasası değil →
§18 sebep kapasite de değil (64× seyreltme kurtarmadı) → **geriye kalan: model
uzun taşımayı hiç öğrenmedi** (eğitim ufku 256).

**Bu test:** Graft hattında işe yarayan mesafe müfredatını küçük modele uygula.
Eğitimde chunk'lar arası taşıma öğretilir (hedef A chunk'ında, sorgu B'de,
aralarında K dolgu chunk'ı; `use_cache=False` olduğu için attention A'yı göremez
— bilgi sadece recurrent state'ten geçer). K müfredatla 0 → 16 büyür, yani
eğitilen taşıma ~4096 token'a çıkar. **Eval §17 ile birebir aynı.**

**ÖN-KAYITLI KRİTERLER** (script docstring'inde de yazılı):
- **BAŞARILI:** 4096 **ve** 16384'te seed-ortalama ≥ %15 → hipotez doğrulandı.
- **KISMİ:** yalnız 4096'da ≥ %15 → müfredat çalışıyor, menzil eğitilen tavanla sınırlı.
- **BAŞARISIZ:** iki uzak gap de < %8 → müfredat açıklaması da reddedilir; sorun mimaride.
- Karşılaştırma tabanı §17 v2: exp {256:15.6, 1024:4.4, 4096:1.1, 16384:5.6},
  cubic {256:18.9, 1024:2.2, 4096:2.2, 16384:2.2} (şans %3.3).

GPU **T4**, Internet **On** → Run All. Süre ~40-60 dk (6 kol; eğitim daha ağır).

In [ ]:
# --- 1. Repo ---
import subprocess, sys, os, re
os.chdir('/kaggle/working')
if not os.path.isdir('HFP'):
    subprocess.run(['git','clone','https://github.com/kayra-hn/HFP.git'], check=True)
else:
    subprocess.run(['git','-C','HFP','pull'], check=True)
os.chdir('/kaggle/working/HFP')
import torch
assert torch.cuda.is_available(), 'GPU secili degil (T4 secin)'
print('GPU:', torch.cuda.get_device_name(0))
assert os.path.exists('review_scripts/carry_curriculum.py'), \
    'carry_curriculum.py yok -> once bilgisayarinizda `git push` yapin!'
ENV = dict(os.environ, PYTHONPATH='/kaggle/working/HFP')
print('hazir.')

In [ ]:
# --- 2. KOSU: 2 mod x 3 seed, tasima mufredati (carry_max=16 ~ 4096 token) ---
raw = {}
for m in ['exp', 'cubic_flux_chunked']:
    for s in ['0', '1', '2']:
        print(f'\n===== {m} | seed {s} =====', flush=True)
        r = subprocess.run([sys.executable, 'review_scripts/carry_curriculum.py', m, s, '1800'],
                           env=dict(ENV, CC_CARRY_MAX='16', CC_STEPS='1200'),
                           check=True, capture_output=True, text=True)
        for line in r.stdout.strip().splitlines():
            if 'step ' in line or 'gap ' in line or 'dogrulama loss' in line or 'FINAL' in line:
                print(line, flush=True)
        mm = re.search(r"FINAL.*?: (\{.*\})", r.stdout)
        if mm:
            raw[(m, s)] = eval(mm.group(1))
print('\nham:', raw)

In [ ]:
# --- 3. TABLO + on-kayitli okuma (§17 v2 tabaniyla yan yana) ---
import statistics as st
BASE = {'exp': {256:15.6, 1024:4.4, 4096:1.1, 16384:5.6},
        'cubic_flux_chunked': {256:18.9, 1024:2.2, 4096:2.2, 16384:2.2}}
GAPS = [256, 1024, 4096, 16384]
print(f'{"gap":>7} | {"exp §17":>8} {"exp §19":>8} | {"cubic §17":>10} {"cubic §19":>10}')
print('-'*54)
new = {}
for m in BASE:
    for g in GAPS:
        vals = [raw[(m, s)][g] for s in ['0','1','2'] if (m, s) in raw and g in raw[(m, s)]]
        new[(m, g)] = st.mean(vals) if vals else float('nan')
for g in GAPS:
    print(f'{g:>7} | {BASE["exp"][g]:>7.1f}% {new[("exp",g)]:>7.1f}% | '
          f'{BASE["cubic_flux_chunked"][g]:>9.1f}% {new[("cubic_flux_chunked",g)]:>9.1f}%')
print('\nSans %3.3 | §17 = mufredatsiz, §19 = tasima mufredatli (bu kosu)')

far = [max(new[('exp',g)], new[('cubic_flux_chunked',g)]) for g in (4096, 16384)]
print(f'\nUzak gapler (4096/16384) en iyi mod: {far[0]:.1f}% / {far[1]:.1f}%')
if far[0] >= 15 and far[1] >= 15:
    print('OKUMA: BASARILI -> "ogrendigi mesafede hatirlar" DOGRULANDI.')
    print('  Cihaz-ici iddia: "O(1) bellek, egitilen tasima mesafesi kadar omur saglar".')
elif far[0] >= 15:
    print('OKUMA: KISMI -> mufredat calisiyor ama menzil egitilen tavanla sinirli.')
    print('  Sonraki: CC_CARRY_MAX buyut (32/64) ve tekrarla.')
elif max(far) < 8:
    print('OKUMA: BASARISIZ -> mufredat aciklamasi da REDDEDILIR.')
    print('  Sonraki eksen mimari: state boyutu (bulk_dim), okuma yolu, dpfp nu.')
else:
    print('OKUMA: Belirsiz bolge (8-15%) — TRIALS artirip tekrarlanmali.')
print('\nBu ciktiyi Claude\'a yapistirin; RESULTS §19 olarak islenecek.')